# Fixing the AudioCraft Environment (phase1)

Your AudioCraft environment suffered the exact same fate as the LTX environment: a rogue package snuck in and upgraded PyTorch to 2.8.0, which broke torchaudio 2.1.0 and completely uninstalled the audiocraft library.

Run these exact commands in your terminal to deep-clean and restore it to the stable 2.1.0 stack:
```bash
conda activate phase1
```
## 1. Nuke the mismatched PyTorch 2.8.0 stack
```bash
pip uninstall -y torch torchvision torchaudio audiocraft xformers
```
## 2. Reinstall the strict, matching 2.1.0 stack required by AudioCraft
```bash
pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118
```
## 3. Install the underlying C-libraries via Conda first
```bash
conda install -c conda-forge "ffmpeg<5" pkg-config av -y
```
## 4. Link the package paths so Pip can see them
```bash
export PKG_CONFIG_PATH="$CONDA_PREFIX/lib/pkgconfig"
```
## 5. Now install AudioCraft (it will instantly find the headers and skip compilation)
```bash
pip install -U audiocraft
```

In [1]:
import core_config

print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")

[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env
[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.


In [2]:
!pip install "numpy<2" "transformers<4.40.0"

  Using cached numpy-1.26.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 51.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 36.3 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:━━━━━━━━━━━━━━━━━━━ 0/3 [numpy]
      Successfully uninstalled numpy-2.0.2━━━━━━ 0/3 [numpy]
  Attempting uninstall: tokenizers━━━━━━━━━━━━━━ 0/3 [numpy]
    Found existing installation: tokenizers 0.22.20/3 [numpy]
    Uninstalling tokenizers-0.22.2:━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [tokenizers]
      Successfully uninstalled tokenizers-0.22.2━━━━━━━━━━━━━━ 1/3 [tokenizers]
  Attempting uninstall: transformers━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [tokenizers]
    Found existing installation: transformers 4.57.6━━━━━━━━━━ 1/3 [tokenizers]
    Uninstalling t

In [3]:
!pip install gradio

In [4]:
import torch
import gradio as gr
from audiocraft.models import AudioGen
from audiocraft.data.audio import audio_write
import tempfile
import os

print("Loading AudioGen Weights (Saving to ~/aireos_models)...")
# Move model to CUDA automatically
model = AudioGen.get_pretrained('facebook/audiogen-medium')
print("AudioGen Loaded Successfully!")

def generate_sfx(prompt: str, duration: int):
    try:
        model.set_generation_params(duration=duration)
        print(f"Generating SFX for prompt: '{prompt}'...")
        
        # AudioGen returns a list of tensors for batches. We pass 1 prompt, we get 1 tensor.
        wav = model.generate([prompt])
        
        # Write to temporary file
        output_dir = os.environ["TMPDIR"]
        temp_path = os.path.join(output_dir, "temp_sfx")
        
        # audio_write adds .wav automatically
        audio_write(temp_path, wav[0].cpu(), model.sample_rate, strategy="loudness")
        
        final_file = f"{temp_path}.wav"
        print(f"Success! Saved to {final_file}")
        return final_file
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        return None

# Build the Gradio UI
with gr.Blocks(title="AudioCraft Testing") as demo:
    gr.Markdown("# AudioCraft (AudioGen): Interactive Testing")
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label="Sound Effect Description", value="cinematic deep bass impact with metallic sci-fi resonance")
            duration = gr.Slider(label="Duration (seconds)", minimum=1, maximum=10, value=5, step=1)
            generate_btn = gr.Button("Generate Audio", variant="primary")
        with gr.Column():
            output_audio = gr.Audio(label="Generated Audio", autoplay=True)

    generate_btn.click(
        fn=generate_sfx,
        inputs=[prompt, duration],
        outputs=[output_audio]
    )

# Binding to 0.0.0.0 and port 7861 so it doesn't conflict with LTX-2.3
demo.launch(server_name="0.0.0.0", server_port=7861, inline=True, share=False)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading AudioGen Weights (Saving to ~/aireos_models)...


compression_state_dict.bin:   0%|          | 0.00/236M [00:00<?, ?B/s]

/nlsasfs/home/lmedge/rahkus/Conda/envs/phase1/lib/python3.9/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


state_dict.bin:   0%|          | 0.00/3.68G [00:00<?, ?B/s]

... (2 outputs left out)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

AudioGen Loaded Successfully!
Running on local URL:  http://0.0.0.0:7861


ValueError: When localhost is not accessible, a shareable link must be created. Please set share=True or check your proxy settings to allow access to localhost.

In [ ]:
demo.close()